# Notebook 05 — Wind Stress over the Southern Ocean

**Kinetic Energy Trends and Eddy Saturation in the Southern Ocean**

---

Computes monthly area-weighted wind stress time series over the ACC and per ocean-basin sector, with Theil–Sen / Modified Mann–Kendall trends. Wind forcing is the driver examined in the eddy saturation analysis (Notebooks 06–07).

**Input:** CMEMS monthly L4 wind stress (`cmems_obs-wind_glo_phy_my_l4_P1M`), streamed via the `copernicusmarine` toolbox; ACC contour mask from `outputs/mean_adt.nc` (Notebook 00).

**Outputs** (to `outputs/trends/`, consumed by Notebooks 06 and 07):

- `wind_stress_timeseries.csv` — monthly whole-ACC τx and |τ|
- `wind_stress_sector_timeseries.csv` — monthly per-sector τx and |τ|
- `wind_stress_trends.csv` — Theil–Sen + MK trend table


## 1. Setup and Imports

In [ ]:
import os
import sys
import warnings
import numpy as np
import xarray as xr
import pandas as pd
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
from cartopy import feature as cfeature
from scipy import stats

warnings.filterwarnings("ignore", category=RuntimeWarning)

# ── Copernicus Marine Toolbox ──────────────────────────────────────────────
try:
    import copernicusmarine
    _CMEMS_AVAILABLE = True
except ImportError:
    _CMEMS_AVAILABLE = False
    print("WARNING: copernicusmarine not installed — remote access unavailable.")
    print("         Install with:  pip install copernicusmarine")

if _CMEMS_AVAILABLE:
    copernicusmarine.login()

REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
sys.path.insert(0, os.path.join(REPO_ROOT, 'scripts'))

from utils.regions_and_masks import compute_acc_mask_from_ssh

print(f"Repository root: {REPO_ROOT}")
print(f"Copernicus Marine Toolbox available: {_CMEMS_AVAILABLE}")
print("Imports OK.")

## 2. Configuration

CMEMS wind product, study region and period, ACC-mask toggle.

In [ ]:
# ── CMEMS Wind product ─────────────────────────────────────────────────────
WIND_DATASET_ID = "cmems_obs-wind_glo_phy_my_l4_P1M"
WIND_VARIABLES  = ["eastward_wind", "northward_wind"]

# ── Region ─────────────────────────────────────────────────────────────────
LONMIN, LONMAX = -180.0, 180.0
LATMIN, LATMAX = -65, -35

# ── Study period (match Notebook 02 / 04) ─────────────────────────────────
YEAR_START = 2000
YEAR_END   = 2022
years      = np.arange(YEAR_START, YEAR_END + 1)

# ── Ocean-basin sectors (Zhang et al. 2021) ───────────────────────────────
ZHANG_SECTORS = {
    "Indian":   (20,  147),
    "Pacific":  (147, 290),
    "Atlantic": (290, 380),
}

# ── ACC SSH-contour mask ───────────────────────────────────────────────────
USE_ACC_MASK  = True
SSH_ACC_SOUTH = -0.6
SSH_ACC_NORTH =  0.2
MEAN_ADT_PATH = os.path.join(REPO_ROOT, 'outputs', 'mean_adt.nc')

# ── Output ─────────────────────────────────────────────────────────────────
OUTPUT_DIR = os.path.join(REPO_ROOT, 'outputs', 'trends')
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("Configuration:")
print(f"  Wind dataset : {WIND_DATASET_ID}")
print(f"  Region       : lon [{LONMIN}, {LONMAX}], lat [{LATMIN}, {LATMAX}]")
print(f"  Period       : {YEAR_START}–{YEAR_END}")
print(f"  ACC mask     : {'ON' if USE_ACC_MASK else 'OFF'}")

## 3. Helper Functions

Sector utilities, Theil–Sen and Modified Mann–Kendall wrappers, running mean.

In [ ]:
def lon_in_sector(lon, lo, hi):

    """Check if longitude (°E, ±180) falls inside [lo, hi), handling wrap."""

    lon360 = lon % 360

    lo360  = lo % 360

    hi360  = hi % 360

    if lo360 < hi360:

        return (lon360 >= lo360) & (lon360 < hi360)

    else:

        return (lon360 >= lo360) | (lon360 < hi360)





def make_sector_mask(lons_2d, sectors):

    """Return dict {sector_name: bool mask (lat, lon)}."""

    masks = {}

    for name, (lo, hi) in sectors.items():

        masks[name] = lon_in_sector(lons_2d, lo, hi)

    return masks





def theil_sen(x, y):

    """Theil-Sen robust slope.  Returns: slope, intercept, lo_slope, hi_slope."""

    mask = np.isfinite(y) & np.isfinite(x)

    if mask.sum() < 3:

        return np.nan, np.nan, np.nan, np.nan

    res = stats.theilslopes(np.asarray(y[mask], float), np.asarray(x[mask], float), alpha=0.95)

    return res.slope, res.intercept, res.low_slope, res.high_slope





def mann_kendall(y):

    """Non-parametric Mann-Kendall test.  Returns: tau, p_value."""

    y = np.asarray(y, dtype=float)

    mask = np.isfinite(y)

    if mask.sum() < 4:

        return np.nan, np.nan

    ym = y[mask]

    n  = len(ym)

    s  = sum(np.sign(ym[j] - ym[k]) for k in range(n-1) for j in range(k+1, n))

    var_s = n * (n - 1) * (2 * n + 5) / 18.0

    z = (s - np.sign(s)) / np.sqrt(var_s) if s != 0 else 0.0

    p = 2 * stats.norm.sf(np.abs(z))

    tau = s / (n * (n - 1) / 2.0)

    return tau, p





def _mk_sigstars(p):

    if not np.isfinite(p):

        return ''

    return '***' if p < 0.001 else ('**' if p < 0.01 else ('*' if p < 0.05 else ''))





def running_mean(series, window=12):

    """12-month centred running mean (consistent with Notebook 03)."""

    return pd.Series(series).rolling(window, center=True, min_periods=window//2).mean().values





print("Helper functions defined (Theil–Sen + Mann–Kendall).")


## 4. ACC Mask

Build the ACC contour mask (−0.6 m to +0.2 m ADT) from Notebook 00's mean ADT and interpolate it to the wind grid.

In [ ]:
if USE_ACC_MASK:
    ds_adt = xr.open_dataset(MEAN_ADT_PATH)
    # Accept either 'adt' or 'sla' as the SSH variable name
    ssh_var = [v for v in ds_adt.data_vars
               if any(k in v.lower() for k in ('adt', 'ssh', 'sla', 'zos'))][0]
    mean_adt = ds_adt[ssh_var].values.squeeze()
    adt_lat  = ds_adt['latitude'].values  if 'latitude'  in ds_adt.coords else ds_adt['lat'].values
    adt_lon  = ds_adt['longitude'].values if 'longitude' in ds_adt.coords else ds_adt['lon'].values
    ds_adt.close()

    acc_mask = compute_acc_mask_from_ssh(
        mean_adt,
        lat       = adt_lat,
        lon       = adt_lon,
        ssh_south = SSH_ACC_SOUTH,
        ssh_north = SSH_ACC_NORTH,
    )
    print(f"ACC mask loaded — {int(acc_mask.sum())} cells inside ACC band")
    print(f"  lat: {acc_mask.latitude.values.min():.2f} to {acc_mask.latitude.values.max():.2f}")
    print(f"  lon: {acc_mask.longitude.values.min():.2f} to {acc_mask.longitude.values.max():.2f}")
else:
    acc_mask = None
    print("ACC mask: DISABLED")


## 5. Retrieve Monthly Wind Stress

Stream monthly-mean τx, τy over the Southern Ocean for the study period from CMEMS.

In [ ]:
assert _CMEMS_AVAILABLE, "copernicusmarine package required"

print(f"Retrieving wind stress from {WIND_DATASET_ID} …")
print(f"  Period: {YEAR_START}-01-01 to {YEAR_END}-12-31")
print(f"  Region: lon [{LONMIN}, {LONMAX}], lat [{LATMIN}, {LATMAX}]")

ds_wind = copernicusmarine.open_dataset(
    dataset_id        = WIND_DATASET_ID,
    minimum_longitude = LONMIN,
    maximum_longitude = LONMAX,
    minimum_latitude  = LATMIN,
    maximum_latitude  = LATMAX,
    start_datetime    = f"{YEAR_START}-01-01",
    end_datetime      = f"{YEAR_END}-12-31",
    variables         = WIND_VARIABLES,
)

print(f"\nDataset loaded:")
print(f"  Time steps : {ds_wind.dims['time']}")
print(f"  Lat points : {ds_wind.dims['latitude']}")
print(f"  Lon points : {ds_wind.dims['longitude']}")
print(ds_wind)

## 6. Wind Stress Fields

Zonal wind stress τx and magnitude |τ| = √(τx² + τy²).

In [ ]:
# Load into memory
ds_wind.load()

tau_x = ds_wind['eastward_wind']     # (time, lat, lon)  N/m²
tau_y = ds_wind['northward_wind']
tau_mag = np.sqrt(tau_x**2 + tau_y**2)
tau_mag.name = 'wind_stress_magnitude'

# Coordinate arrays
lats = ds_wind['latitude'].values
lons = ds_wind['longitude'].values
times = pd.to_datetime(ds_wind['time'].values)

print(f"τx  range: {float(tau_x.min()):.4f}  to {float(tau_x.max()):.4f} N/m²")
print(f"|τ| range: {float(tau_mag.min()):.4f} to {float(tau_mag.max()):.4f} N/m²")
print(f"Time: {times[0].strftime('%Y-%m')} to {times[-1].strftime('%Y-%m')} ({len(times)} months)")

## 7. Area-Weighted Monthly Time Series

cos(latitude)-weighted mean τx and |τ| inside the ACC mask → `df_ts`.

In [ ]:
# ── Area weights ───────────────────────────────────────────────────────────
cos_lat = np.cos(np.deg2rad(lats))
weights_2d = np.broadcast_to(cos_lat[:, None], (len(lats), len(lons)))

# ── ACC mask (interpolated to wind grid if needed) ─────────────────────────
if acc_mask is not None:
    # Cast to float first (interp doesn't support bool), then threshold back
    acc_on_wind = (
        acc_mask.astype(float)
        .interp(latitude=lats, longitude=lons, method='nearest')
        .values > 0.5
    )
    spatial_mask = acc_on_wind  # True = inside ACC
    print(f"ACC mask interpolated to wind grid: {spatial_mask.sum()} / {spatial_mask.size} cells")
else:
    spatial_mask = np.ones((len(lats), len(lons)), dtype=bool)

# ── Masked area weights ────────────────────────────────────────────────────
w_masked = np.where(spatial_mask, weights_2d, 0.0)
w_sum    = w_masked.sum()

# ── Time series: area-weighted mean τx and |τ| ─────────────────────────────
n_times = len(times)
ts_tau_x   = np.full(n_times, np.nan)
ts_tau_mag = np.full(n_times, np.nan)

for t in range(n_times):
    tx_t = tau_x.values[t]    # (lat, lon)
    tm_t = tau_mag.values[t]
    # mask NaN ocean cells
    valid = np.isfinite(tx_t) & spatial_mask
    w = np.where(valid, weights_2d, 0.0)
    ws = w.sum()
    if ws > 0:
        ts_tau_x[t]   = np.nansum(tx_t * w) / ws
        ts_tau_mag[t] = np.nansum(tm_t * w) / ws

# Build a DataFrame
df_ts = pd.DataFrame({
    'time':    times,
    'year':    times.year,
    'month':   times.month,
    'tau_x':   ts_tau_x,
    'tau_mag': ts_tau_mag,
})

print(f"Time series length: {len(df_ts)} months")
print(f"Mean τx  : {df_ts['tau_x'].mean():.4f} N/m²")
print(f"Mean |τ| : {df_ts['tau_mag'].mean():.4f} N/m²")
df_ts.head()


## 8. Per-Sector Time Series

Same area-weighted means restricted to the Atlantic, Indian and Pacific sectors.

In [ ]:
LONS_2D, LATS_2D = np.meshgrid(lons, lats)

sector_masks = make_sector_mask(LONS_2D, ZHANG_SECTORS)

sector_names = list(ZHANG_SECTORS.keys())

sector_colors = {'Indian': '#2ca02c', 'Pacific': '#1f77b4', 'Atlantic': '#d62728'}



sector_ts = {}



for sname in sector_names:

    smask = sector_masks[sname]

    if acc_mask is not None:

        smask = smask & acc_on_wind



    w = np.where(smask, weights_2d, 0.0)

    ws = w.sum()



    tx_ts  = np.full(n_times, np.nan)

    tm_ts  = np.full(n_times, np.nan)

    for t in range(n_times):

        valid = np.isfinite(tau_x.values[t]) & smask

        wv = np.where(valid, weights_2d, 0.0)

        wsv = wv.sum()

        if wsv > 0:

            tx_ts[t] = np.nansum(tau_x.values[t] * wv) / wsv

            tm_ts[t] = np.nansum(tau_mag.values[t] * wv) / wsv



    sector_ts[sname] = {'tau_x': tx_ts, 'tau_mag': tm_ts}

print('Sector series computed:', ', '.join(sector_names))


## 9. Save Results

Write the monthly time series (whole-ACC and per-sector) and the trend table to `outputs/trends/`.

In [ ]:
out_dir = os.path.join(REPO_ROOT, 'outputs', 'trends')

os.makedirs(out_dir, exist_ok=True)

out_csv = os.path.join(out_dir, 'wind_stress_trends.csv')

# ── Monthly whole-ACC time series (consumed by Notebook 06) ────────────────
out_ts = os.path.join(out_dir, 'wind_stress_timeseries.csv')
df_ts.to_csv(out_ts, index=False)
print(f"Saved: {out_ts}")

# ── Monthly per-sector time series (consumed by Notebook 07) ───────────────
out_sector = os.path.join(out_dir, 'wind_stress_sector_timeseries.csv')
_rows = []
for sname in sector_names:
    _rows.append(pd.DataFrame({
        'year': df_ts['year'].values,
        'month': df_ts['month'].values,
        'sector': sname,
        'tau_x': sector_ts[sname]['tau_x'],
        'tau_mag': sector_ts[sname]['tau_mag'],
    }))
pd.concat(_rows, ignore_index=True).to_csv(out_sector, index=False)
print(f"Saved: {out_sector}")




rows = []



def _annual_trend(df, col):

    ann = df.groupby('year')[col].mean()

    x = ann.index.values.astype(float)

    y = ann.values.astype(float)

    slope, intercept, lo, hi = theil_sen(x, y)

    mk_tau, mk_p = mann_kendall(y)

    return slope, lo, hi, mk_tau, mk_p



# Whole ACC trends (area-weighted TS computed earlier)

for col, label in [('tau_x', 'tau_x'), ('tau_mag', 'tau_mag')]:

    slope, lo, hi, mk_tau, mk_p = _annual_trend(df_ts, col)

    rows.append({

        'sector': 'Whole ACC',

        'variable': label,

        'theil_sen_slope_per_year': slope,

        'theil_sen_ci95_low_per_year': lo,

        'theil_sen_ci95_high_per_year': hi,

        'mann_kendall_tau': mk_tau,

        'mann_kendall_p': mk_p,

        'significant_05': bool(np.isfinite(mk_p) and (mk_p < 0.05)),

    })



# Per-sector trends

for sname in sector_names:

    for key, label in [('tau_x', 'tau_x'), ('tau_mag', 'tau_mag')]:

        tmp = pd.DataFrame({'year': df_ts['year'].values,

                            'month': df_ts['month'].values,

                            key: sector_ts[sname][key]})

        slope, lo, hi, mk_tau, mk_p = _annual_trend(tmp, key)

        rows.append({

            'sector': sname,

            'variable': label,

            'theil_sen_slope_per_year': slope,

            'theil_sen_ci95_low_per_year': lo,

            'theil_sen_ci95_high_per_year': hi,

            'mann_kendall_tau': mk_tau,

            'mann_kendall_p': mk_p,

            'significant_05': bool(np.isfinite(mk_p) and (mk_p < 0.05)),

        })



df_trends = pd.DataFrame(rows)

df_trends.to_csv(out_csv, index=False)



print(f"Saved: {out_csv}")

display(df_trends)


## Summary

Monthly ACC and sector wind stress series with non-parametric trends, saved for the Drake Passage analysis (Notebooks 06–07).
